# Data Quality Audit

## Objective

This notebook evaluates the quality of the Orders, Returns, and People datasets before data cleaning and analysis.

The audit will check:

- Dataset structure and data types
- Missing values
- Duplicate records
- Invalid quantities and prices
- Date formats
- Sales, profit, and discount fields
- Customer and order identifier consistency
- Returns and regional mapping quality

In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
project_root = current_folder.parent
raw_data_folder = project_root / "data" / "raw"

print("Notebook folder:", current_folder.name)
print("Project folder:", project_root.name)
print(
    "Raw data folder:",
    raw_data_folder.relative_to(project_root)
)
print("Raw folder exists:", raw_data_folder.exists())

Notebook folder: notebooks
Project folder: ecommerce_sales_customer_profitability_analytics
Raw data folder: data\raw
Raw folder exists: True


In [2]:
orders_path = raw_data_folder / "orders_data.csv"
returns_path = raw_data_folder / "returns_data.csv"
people_path = raw_data_folder / "people_data.csv"

orders = pd.read_csv(orders_path,
                    dtype={"Postal Code" : "str"}
                   )
returns = pd.read_csv(returns_path)

people = pd.read_csv(people_path)

print("Orders:", orders.shape)
print("Returns:", returns.shape)
print("People:", people.shape)

Orders: (51290, 24)
Returns: (1079, 3)
People: (24, 2)


## Dataset Grain

- Orders contains one row per product line within an order.
- Returns contains one row per returned Order ID.
- People contains one row per person-region assignment.

All row counts and KPIs in this notebook are interpreted according to these dataset grains.

## 1. Dataset Structure and Data Types

I inspect the dataset structure, column names, non-null counts, and current data types before cleaning.

In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  str    
 2   Order Date      51290 non-null  str    
 3   Ship Date       51290 non-null  str    
 4   Ship Mode       51290 non-null  str    
 5   Customer ID     51290 non-null  str    
 6   Customer Name   51290 non-null  str    
 7   Segment         51290 non-null  str    
 8   Postal Code     9994 non-null   str    
 9   City            51290 non-null  str    
 10  State           51290 non-null  str    
 11  Country         51290 non-null  str    
 12  Region          51290 non-null  str    
 13  Market          51290 non-null  str    
 14  Product ID      51290 non-null  str    
 15  Category        51290 non-null  str    
 16  Sub-Category    51290 non-null  str    
 17  Product Name    51290 non-null  str    
 1

In [4]:
orders[
    ["Order Date", "Ship Date", "Sales", "Profit"]
].head(10)

,Order Date,Ship Date,Sales,Profit
0,11-11-14,13-11-14,$221.98,$62.15
1,05-02-14,07-02-14,"$3,709.40",-$288.77
2,17-10-14,18-10-14,"$5,175.17",$919.97
3,28-01-14,30-01-14,"$2,892.51",-$96.54
4,05-11-14,06-11-14,"$2,832.96",$311.52
5,28-06-14,01-07-14,"$2,862.68",$763.28
6,06-11-12,08-11-12,"$1,822.08",$564.84
7,14-04-13,18-04-13,"$5,244.84",$996.48
8,11-11-14,13-11-14,$341.96,$54.71
9,06-03-12,07-03-12,$48.71,$5.48


### 1.1 Date Format Validation

I validate whether the order and shipping dates follow the expected day-month-year format without modifying the raw columns.

In [5]:
order_date_test = pd.to_datetime(
    orders["Order Date"],
    format = "%d-%m-%y",
    errors = "coerce"
)

ship_date_test = pd.to_datetime(
    orders["Ship Date"],
    format = "%d-%m-%y",
    errors = "coerce"
)

print("Invalid order date values:", order_date_test.isna().sum())
print("Invalid ship date values:", ship_date_test.isna().sum())

Invalid order date values: 0
Invalid ship date values: 0


### 1.2 Currency Format Validation

I test whether the Sales and Profit values can be converted to numeric values after removing currency symbols and thousands separators.

In [6]:
sales_test = pd.to_numeric(
    orders["Sales"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors = "coerce"
)

profit_test = pd.to_numeric(
    orders["Profit"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors = "coerce"
)
print("Invalid Sales values:", sales_test.isna().sum())
print("Invalid Profit values:", profit_test.isna().sum())

Invalid Sales values: 0
Invalid Profit values: 0


## 2. Missing-Value Audit

I calculate the number and percentage of missing values in each Orders column to determine which fields require cleaning or business-rule decisions.

In [7]:
orders_missing = pd.DataFrame({
    "Missing Percentage": (orders.isna().mean() * 100).round(2)
})

# orders_missing = orders_missing[
#     orders_missing["Missing Count"] > 0
# ].sort_values(
#     by="Missing Percentage",
#     ascending=False
# )

display(orders_missing)

,Missing Percentage
Row ID,0.00
Order ID,0.00
Order Date,0.00
Ship Date,0.00
Ship Mode,0.00
Customer ID,0.00
Customer Name,0.00
Segment,0.00
Postal Code,80.51
City,0.00


### 2.1 Supporting Dataset Structure

I inspect the Returns and People datasets before checking their missing values and key relationships.

In [8]:
print("RETURNS DATASET")
returns.info()

print("\nPEOPLE DATASET")
people.info()

RETURNS DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1079 entries, 0 to 1078
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Returned  1079 non-null   str  
 1   Order ID  1079 non-null   str  
 2   Region    1079 non-null   str  
dtypes: str(3)
memory usage: 66.2 KB

PEOPLE DATASET
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Person  24 non-null     str  
 1   Region  24 non-null     str  
dtypes: str(2)
memory usage: 1.1 KB


## 3. Duplicate-Record Audit

I check each dataset for completely duplicated rows before evaluating duplicates in business identifiers such as Row ID and Order ID.

In [9]:
print("Exact duplicate rows in Orders:", orders.duplicated().sum())
print("Exact duplicate rows in Returns:", returns.duplicated().sum())
print("Exact duplicate rows in People:", people.duplicated().sum())

Exact duplicate rows in Orders: 0
Exact duplicate rows in Returns: 0
Exact duplicate rows in People: 0


### 3.1 Row ID Uniqueness

I validate whether Row ID uniquely identifies every sales line in the Orders dataset.

In [10]:
print("Total Orders rows:", len(orders))
print("Unique Row IDs:", orders["Row ID"].nunique())
print("Duplicated Row IDs:", orders["Row ID"].duplicated().sum())

Total Orders rows: 51290
Unique Row IDs: 51290
Duplicated Row IDs: 0


### 3.2 Order-Level Line Counts

I examine how many sales lines belong to each Order ID. Repeated Order IDs are expected when a single order contains multiple products.

In [11]:
order_line_counts = orders.groupby("Order ID").size()

print("Total sales lines:", len(orders))
print("Unique Order IDs:", orders["Order ID"].nunique())
print(
    "Orders containing multiple sales lines:",
    (order_line_counts > 1).sum()
)
print("Maximum lines in one Order ID:", order_line_counts.max())

Total sales lines: 51290
Unique Order IDs: 25728
Orders containing multiple sales lines: 12793
Maximum lines in one Order ID: 14


### 3.3 Order ID Consistency

I test whether each Order ID is consistently associated with one customer, one order date, one market, and one region.

In [12]:
order_id_check = orders.groupby("Order ID").agg(
    customer_id_count=("Customer ID", "nunique"),
    order_date_count=("Order Date", "nunique"),
    market_count=("Market", "nunique"),
    region_count=("Region", "nunique")
)

order_id_issues = order_id_check[
    (order_id_check["customer_id_count"] > 1)
    | (order_id_check["order_date_count"] > 1)
    | (order_id_check["market_count"] > 1)
    | (order_id_check["region_count"] > 1)
]

orders_with_multiple_customers = order_id_check[
    order_id_check["customer_id_count"] > 1
]

print("Order IDs with consistency issues:", len(order_id_issues))
display(order_id_issues.head(20))

Order IDs with consistency issues: 11


,customer_id_count,order_date_count,market_count,region_count
Order ID,,,,
CA-2012-CS12355140-41135,2,1,1,2
CA-2013-EM13960140-41548,2,1,1,2
CA-2013-KB16240140-41585,2,1,1,2
CA-2013-PJ18835140-41592,2,1,1,2
CA-2014-BW11200140-41940,2,1,1,2
CA-2014-GT14710140-41689,2,1,1,2
CA-2014-JE15715140-41999,2,1,1,2
CA-2014-JG15160140-41979,2,1,1,2
CA-2015-AG10330140-42361,2,1,1,2


In [13]:
multiple_customer_order_details = orders.loc[
    orders["Order ID"].isin(
        orders_with_multiple_customers.index
    ),
    [
        "Order ID",
        "Order Date",
        "Customer ID",
        "Customer Name",
        "Country",
        "Region",
        "Market"
    ]
].drop_duplicates().sort_values(
    by=["Order ID", "Customer ID", "Region"]
)

display(multiple_customer_order_details)

,Order ID,Order Date,Customer ID,Customer Name,Country,Region,Market
10947,CA-2012-CS12355140-41135,14-08-12,CS-123551406,Christine Sundaresam,United States,Eastern US,USCA
10914,CA-2012-CS12355140-41135,14-08-12,CS-123551408,Christine Sundaresam,United States,Southern US,USCA
19909,CA-2013-EM13960140-41548,01-10-13,EM-139601402,Eric Murdock,United States,Central US,USCA
19893,CA-2013-EM13960140-41548,01-10-13,EM-139601404,Eric Murdock,United States,Western US,USCA
31371,CA-2013-KB16240140-41585,07-11-13,KB-162401404,Karen Bern,United States,Western US,USCA
31395,CA-2013-KB16240140-41585,07-11-13,KB-162401406,Karen Bern,United States,Eastern US,USCA
42722,CA-2013-PJ18835140-41592,14-11-13,PJ-188351406,Patrick Jones,United States,Eastern US,USCA
42715,CA-2013-PJ18835140-41592,14-11-13,PJ-188351408,Patrick Jones,United States,Southern US,USCA
5741,CA-2014-BW11200140-41940,28-10-14,BW-112001406,Ben Wallace,United States,Eastern US,USCA
5752,CA-2014-BW11200140-41940,28-10-14,BW-112001408,Ben Wallace,United States,Southern US,USCA


In [14]:
composite_order_key_test = (
    orders["Order ID"].astype("string")
    + "|"
    + orders["Customer ID"].astype("string")
)

print("Unique raw Order IDs:", orders["Order ID"].nunique())
print(
    "Unique Order ID + Customer ID combinations:",
    composite_order_key_test.nunique()
)

Unique raw Order IDs: 25728
Unique Order ID + Customer ID combinations: 25739


### 3.4 Composite Order Key Validation

I test an Order ID and Customer ID composite key to prevent distinct customer-region order instances from being merged.

In [15]:
composite_key_check = orders.assign(
    Order_Key_Test=composite_order_key_test
).groupby(
    "Order_Key_Test"
).agg(
    order_date_count=("Order Date", "nunique"),
    region_count=("Region", "nunique"),
    market_count=("Market", "nunique")
)

composite_key_issues = composite_key_check[
    (composite_key_check["order_date_count"] > 1)
    | (composite_key_check["region_count"] > 1)
    | (composite_key_check["market_count"] > 1)
]

print(
    "Composite keys with consistency issues:",
    len(composite_key_issues)
)

Composite keys with consistency issues: 0


#### Audit Finding: Order Identifier

I found 11 raw Order IDs associated with two different customer identifiers and regions. These records shared the same customer name, order date, country, and market, but represented distinct customer-region combinations.

- Unique raw Order IDs: 25,728
- Unique Order ID and Customer ID combinations: 25,739
- Composite keys with date, region, or market inconsistencies: 0

For this project, I will retain the original Order ID and create an analytical Order Key by combining Order ID and Customer ID. This prevents 11 distinct customer-region order instances from being merged during order-level analysis.

This rule is treated as a documented analytical assumption because confirmation from the source-system owner is unavailable.

## 4. Customer Identifier Audit

I compare Customer ID and Customer Name to determine whether customer records are fragmented across multiple identifiers.

In [16]:
customer_ids_per_name = (
    orders.groupby("Customer Name")["Customer ID"].nunique()
)

print("Unique Customer IDs:", orders["Customer ID"].nunique())
print("Unique Customer Names:", orders["Customer Name"].nunique())
print(
    "Customer names linked to multiple IDs:",
    (customer_ids_per_name > 1).sum()
)
print(
    "Maximum IDs linked to one customer name:",
    customer_ids_per_name.max()
)

Unique Customer IDs: 17415
Unique Customer Names: 796
Customer names linked to multiple IDs: 795
Maximum IDs linked to one customer name: 35


In [17]:
customer_base_id_test = orders["Customer ID"].str[:8]

print(
    "Unique raw Customer IDs:",
    orders["Customer ID"].nunique()
)
print(
    "Unique first-eight-character customer IDs:",
    customer_base_id_test.nunique()
)
print(
    "Unique Customer Names:",
    orders["Customer Name"].nunique()
)

Unique raw Customer IDs: 17415
Unique first-eight-character customer IDs: 3416
Unique Customer Names: 796


In [18]:
base_ids_per_name = pd.DataFrame({
    "Customer Name": orders["Customer Name"],
    "Base ID Test": customer_base_id_test
}).groupby(
    "Customer Name"
)["Base ID Test"].nunique()

print(
    "Names linked to multiple base-ID tests:",
    (base_ids_per_name > 1).sum()
)
print(
    "Maximum base-ID tests linked to one name:",
    base_ids_per_name.max()
)

display(
    base_ids_per_name.sort_values(
        ascending=False
    ).head(10)
)

Names linked to multiple base-ID tests: 790
Maximum base-ID tests linked to one name: 12


Customer Name
Alejandro Savely    12
Anthony Jacobs      11
Andy Yotov          10
Alyssa Tate         10
Anna Andreadi        9
Art Ferguson         9
Annie Zypern         9
Andy Reiter          9
Anthony Johnson      9
Allen Armold         9
Name: Base ID Test, dtype: int64

### 4.1 Customer Identity Consistency

I validate the relationships between Customer ID, Customer Name, and Segment before selecting an analytical customer key.

In [19]:
names_per_customer_id = (
    orders.groupby("Customer ID")["Customer Name"].nunique()
)

segments_per_customer_name = (
    orders.groupby("Customer Name")["Segment"].nunique()
)

print(
    "Customer IDs linked to multiple names:",
    (names_per_customer_id > 1).sum()
)
print(
    "Customer names linked to multiple segments:",
    (segments_per_customer_name > 1).sum()
)

Customer IDs linked to multiple names: 0
Customer names linked to multiple segments: 0


#### Audit Finding: Customer Identifier

The dataset contains 17,415 raw Customer IDs but only 796 distinct Customer Names. Of the 796 names, 795 are associated with multiple raw Customer IDs, with a maximum of 35 IDs for one name.

A fixed first-eight-character ID rule was tested and rejected because it produced 3,416 identifiers and remained inconsistent for 790 customer names.

Each raw Customer ID maps to only one Customer Name, and each Customer Name maps to only one Segment.

For customer-level analytics, I will create a surrogate Customer Key from the 796 distinct Customer Names while retaining the original Customer ID as a source account identifier. This prevents customer activity from being fragmented across geography-specific IDs.

This is a dataset-specific analytical rule. In a real production system, a master customer identifier or additional fields such as email would be required to distinguish people who share the same name.

## 5. Numeric Validity Audit

I test numeric business fields against valid ranges while treating negative profit as a business outcome rather than a data error.

In [20]:
print(
    "Non-positive Quantity values:",
    (orders["Quantity"] <= 0).sum()
)

print(
    "Non-positive Sales values:",
    (sales_test <= 0).sum()
)

print(
    "Discount values outside 0 to 1:",
    (
        (orders["Discount"] < 0)
        | (orders["Discount"] > 1)
    ).sum()
)

print(
    "Negative Shipping Cost values:",
    (orders["Shipping Cost"] < 0).sum()
)

print(
    "Negative Profit transactions:",
    (profit_test < 0).sum()
)

print(
    "Zero Profit transactions:",
    (profit_test == 0).sum()
)

Non-positive Quantity values: 0
Non-positive Sales values: 0
Discount values outside 0 to 1: 0
Negative Shipping Cost values: 0
Negative Profit transactions: 12540
Zero Profit transactions: 672


In [21]:
numeric_range_summary = pd.DataFrame({
    "Metric": [
        "Quantity",
        "Sales",
        "Discount",
        "Profit",
        "Shipping Cost"
    ],
    "Minimum": [
        orders["Quantity"].min(),
        sales_test.min(),
        orders["Discount"].min(),
        profit_test.min(),
        orders["Shipping Cost"].min()
    ],
    "Maximum": [
        orders["Quantity"].max(),
        sales_test.max(),
        orders["Discount"].max(),
        profit_test.max(),
        orders["Shipping Cost"].max()
    ]
})

display(numeric_range_summary)

,Metric,Minimum,Maximum
0,Quantity,1.000,14.00
1,Sales,0.440,22638.48
2,Discount,0.000,0.85
3,Profit,-6599.980,8399.98
4,Shipping Cost,1.002,933.57


## 6. Outlier Audit

I use the IQR method to flag unusually low or high numeric values. Outlier flags are used for investigation and are not treated as automatic deletion rules.

In [22]:
outlier_fields = {
    "Quantity": orders["Quantity"],
    "Sales": sales_test,
    "Discount": orders["Discount"],
    "Profit": profit_test,
    "Shipping Cost": orders["Shipping Cost"]
}

outlier_results = []

for metric, values in outlier_fields.items():
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outlier_mask = (
        (values < lower_bound)
        | (values > upper_bound)
    )

    outlier_results.append({
        "Metric": metric,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outlier Count": int(outlier_mask.sum()),
        "Outlier Percentage": round(
            outlier_mask.mean() * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary)

,Metric,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage
0,Quantity,-2.50,9.50,877,1.71
1,Sales,-299.68,581.49,5655,11.03
2,Discount,-0.30,0.50,4172,8.13
3,Profit,-55.22,92.02,9755,19.02
4,Shipping Cost,-30.15,57.21,5909,11.52


#### Outlier Treatment Decision

The IQR method flagged 877 Quantity values, 5,655 Sales values, 4,172 Discount values, 9,755 Profit values, and 5,909 Shipping Cost values as statistical outliers.

These records satisfy the valid business ranges and may represent large orders, high discounts, major losses, or expensive shipments. I will retain them in the cleaned dataset and investigate their business impact during exploratory and profitability analysis.

I will not cap or remove outliers when calculating revenue, profit, or operational KPIs because doing so would distort actual company performance.

## 7. Date-Logic Audit

I validate the chronological relationship between Order Date and Ship Date and calculate the observed shipping duration.

In [23]:
shipping_days_test = (
    ship_date_test - order_date_test
).dt.days

print(
    "Ship Date before Order Date:",
    (shipping_days_test < 0).sum()
)
print(
    "Same-day sales lines:",
    (shipping_days_test == 0).sum()
)
print(
    "Minimum shipping days:",
    shipping_days_test.min()
)
print(
    "Maximum shipping days:",
    shipping_days_test.max()
)

Ship Date before Order Date: 0
Same-day sales lines: 2600
Minimum shipping days: 0
Maximum shipping days: 7


In [24]:
shipping_consistency_check = pd.DataFrame({
    "Order Key": composite_order_key_test,
    "Ship Date": ship_date_test,
    "Ship Mode": orders["Ship Mode"],
    "Shipping Days": shipping_days_test
}).groupby(
    "Order Key"
).agg(
    ship_date_count=("Ship Date", "nunique"),
    ship_mode_count=("Ship Mode", "nunique"),
    shipping_days_count=("Shipping Days", "nunique")
)

orders_with_split_shipping = shipping_consistency_check[
    (shipping_consistency_check["ship_date_count"] > 1)
    | (shipping_consistency_check["ship_mode_count"] > 1)
    | (shipping_consistency_check["shipping_days_count"] > 1)
]

print(
    "Order Keys with multiple shipping patterns:",
    len(orders_with_split_shipping)
)

display(orders_with_split_shipping)

Order Keys with multiple shipping patterns: 11


,ship_date_count,ship_mode_count,shipping_days_count
Order Key,,,
ES-2013-CD1192045-41609|CD-1192045,2,1,2
ES-2013-JF1541545-41509|JF-1541545,1,2,1
ES-2015-BG1174045-42353|BG-1174045,2,2,2
ES-2015-TM21490139-42223|TM-21490139,2,2,2
IN-2013-DM133457-41306|DM-133457,2,2,2
IN-2015-RF1984058-42273|RF-1984058,2,2,2
MX-2014-TA2138582-41991|TA-2138582,2,2,2
MX-2015-CC1210082-42318|CC-1210082,2,1,2
NI-2014-TC1153595-41823|TC-1153595,2,2,2


#### Audit Finding: Shipping Dates and Modes

No sales line has a Ship Date earlier than its Order Date. The observed shipping duration ranges from 0 to 7 days.

I found 11 analytical Order Keys with more than one shipping pattern. These orders contain multiple Ship Dates, Ship Modes, or shipping durations across their product lines.

I will retain the line-level shipping information and treat these records as potential split shipments rather than duplicate errors. Order-level KPIs will use the analytical Order Key, while shipping analysis will respect the Ship Date and Ship Mode stored on each sales line.

## 8. Returns Dataset Audit

I validate the return-status values and determine whether each Returns row uniquely identifies a returned order.

In [25]:
print("Return-status values:")
display(
    returns["Returned"].value_counts(
        dropna=False
    )
)

print("Total Returns rows:", len(returns))
print(
    "Unique returned Order IDs:",
    returns["Order ID"].nunique()
)
print(
    "Duplicated returned Order IDs:",
    returns["Order ID"].duplicated().sum()
)
print(
    "Unique return regions:",
    returns["Region"].nunique()
)

Return-status values:


Returned
Yes    1079
Name: count, dtype: int64

Total Returns rows: 1079
Unique returned Order IDs: 1079
Duplicated returned Order IDs: 0
Unique return regions: 24


In [26]:
return_id_match = returns["Order ID"].isin(
    orders["Order ID"]
)

print(
    "Return IDs found in Orders:",
    return_id_match.sum()
)
print(
    "Return IDs not found in Orders:",
    (~return_id_match).sum()
)

returned_collision_ids = returns.loc[
    returns["Order ID"].isin(
        orders_with_multiple_customers.index
    ),
    ["Returned", "Order ID", "Region"]
]

print(
    "Returned IDs affected by Order ID collisions:",
    len(returned_collision_ids)
)

display(returned_collision_ids)

Return IDs found in Orders: 1079
Return IDs not found in Orders: 0
Returned IDs affected by Order ID collisions: 2


,Returned,Order ID,Region
101,Yes,CA-2013-EM13960140-41548,Western US
956,Yes,CA-2015-HM14860140-42221,Western US


### 8.1 Returns Join-Key Validation

I test Order ID and Region as a composite join key to prevent returned orders from matching unrelated regional order records.

In [27]:
order_return_keys = orders[
    ["Order ID", "Region"]
].drop_duplicates()

return_join_test = returns.merge(
    order_return_keys,
    on=["Order ID", "Region"],
    how="left",
    indicator=True
)

print("Return join results:")
display(
    return_join_test["_merge"].value_counts(
        dropna=False
    )
)

print("Unmatched return records:")
display(
    return_join_test.loc[
        return_join_test["_merge"] != "both",
        ["Returned", "Order ID", "Region"]
    ]
)

Return join results:


_merge
both          1073
left_only        6
right_only       0
Name: count, dtype: int64

Unmatched return records:


,Returned,Order ID,Region
38,Yes,CA-2014-SS1041023-41995,Eastern Canada
240,Yes,CA-2012-CC247523-41031,Eastern Canada
572,Yes,CA-2013-NF847523-41549,Western Canada
697,Yes,CA-2014-TZ1144523-41985,Eastern Canada
897,Yes,CA-2015-JS594023-42346,Eastern Canada
941,Yes,CA-2012-RA988523-41160,Eastern Canada


In [28]:
unmatched_return_ids = return_join_test.loc[
    return_join_test["_merge"] == "left_only",
    "Order ID"
]

unmatched_order_regions = orders.loc[
    orders["Order ID"].isin(unmatched_return_ids),
    [
        "Order ID",
        "Country",
        "Region",
        "Market"
    ]
].drop_duplicates().sort_values(
    by="Order ID"
)

display(unmatched_order_regions)

,Order ID,Country,Region,Market
11565,CA-2012-CC247523-41031,Canada,Canada,USCA
45718,CA-2012-RA988523-41160,Canada,Canada,USCA
41604,CA-2013-NF847523-41549,Canada,Canada,USCA
47471,CA-2014-SS1041023-41995,Canada,Canada,USCA
50077,CA-2014-TZ1144523-41985,Canada,Canada,USCA
29860,CA-2015-JS594023-42346,Canada,Canada,USCA


### 8.2 Canada Region Standardization

I standardize Eastern Canada and Western Canada to Canada for the return join because the Orders dataset stores Canadian records at the national region level.

In [29]:
returns_join_ready_test = returns.copy()

returns_join_ready_test["Standardized Region"] = (
    returns_join_ready_test["Region"].replace({
        "Eastern Canada": "Canada",
        "Western Canada": "Canada"
    })
)

order_keys_for_return_join = (
    orders[["Order ID", "Region"]]
    .drop_duplicates()
    .rename(
        columns={"Region": "Standardized Region"}
    )
)

return_join_retest = returns_join_ready_test.merge(
    order_keys_for_return_join,
    on=["Order ID", "Standardized Region"],
    how="left",
    indicator=True
)

display(
    return_join_retest["_merge"].value_counts(
        dropna=False
    )
)

_merge
both          1079
left_only        0
right_only       0
Name: count, dtype: int64

#### Audit Finding: Returns Join

The Returns dataset contains 1,079 unique Order IDs, and every record has a Returned status of Yes.

Two returned Order IDs are affected by raw Order ID collisions, so joining Returns to Orders on Order ID alone is unsafe.

An Order ID and Region join matched 1,073 records. The remaining six records used Eastern Canada or Western Canada in Returns, while Orders stored their Region as Canada. After standardizing these two labels to Canada, all 1,079 return records matched successfully.

For cleaning, I will retain the original return-region field, create a standardized return region, and join Returns to Orders using Order ID and standardized Region.

The Returns file operates at order level and does not identify individual returned products or quantities. Therefore, product analysis will describe products contained in returned orders unless a full-order-return assumption is explicitly stated.

## 9. People Dataset Audit

I validate whether the People dataset provides a unique person for every sales region and whether its region labels match the Orders dataset.

In [30]:
print(
    "Unique Orders regions:",
    orders["Region"].nunique()
)
print(
    "Unique People regions:",
    people["Region"].nunique()
)
print(
    "Duplicated People regions:",
    people["Region"].duplicated().sum()
)

orders_regions = set(orders["Region"])
people_regions = set(people["Region"])

print(
    "Regions present only in Orders:",
    sorted(orders_regions - people_regions)
)
print(
    "Regions present only in People:",
    sorted(people_regions - orders_regions)
)

Unique Orders regions: 23
Unique People regions: 24
Duplicated People regions: 0
Regions present only in Orders: ['Canada']
Regions present only in People: ['Eastern Canada', 'Western Canada']


In [31]:
canada_people = people.loc[
    people["Region"].isin(
        ["Eastern Canada", "Western Canada"]
    ),
    ["Person", "Region"]
]

display(canada_people)

,Person,Region
7,Lynne Marchand,Eastern Canada
21,Angela Jephson,Western Canada


In [32]:
canada_sales_mask = orders["Region"] == "Canada"

print(
    "Canada sales lines:",
    canada_sales_mask.sum()
)
print(
    "Canada analytical orders:",
    composite_order_key_test[
        canada_sales_mask
    ].nunique()
)
print(
    "Canada revenue:",
    round(
        sales_test[canada_sales_mask].sum(),
        2
    )
)

Canada sales lines: 384
Canada analytical orders: 205
Canada revenue: 66928.17


#### Audit Finding: People and Region Mapping

The People dataset contains one person for each of its 24 region labels, while Orders contains 23 region labels.

Twenty-two sales regions match directly. Orders stores all Canadian sales under Canada, while People divides responsibility between Eastern Canada and Western Canada.

This mismatch affects 384 sales lines, 205 analytical orders, and $66,928.17 in revenue. The Orders dataset does not contain a direct field that identifies which Canadian manager is responsible for each record.

I will retain the original People data and avoid assigning Canadian performance to either manager without an approved geographic mapping rule. Manager-level analysis will label Canada as unassigned or exclude it with a disclosed limitation, while general regional analysis will continue to include Canada.

## 10. Product Identifier Audit

I validate whether Product ID provides a consistent one-to-one product mapping for the future DimProduct table.

In [33]:
product_id_check = orders.groupby(
    "Product ID"
).agg(
    product_name_count=("Product Name", "nunique"),
    category_count=("Category", "nunique"),
    subcategory_count=("Sub-Category", "nunique")
)

product_id_issues = product_id_check[
    (product_id_check["product_name_count"] > 1)
    | (product_id_check["category_count"] > 1)
    | (product_id_check["subcategory_count"] > 1)
]

product_ids_per_name = orders.groupby(
    "Product Name"
)["Product ID"].nunique()

print(
    "Unique Product IDs:",
    orders["Product ID"].nunique()
)
print(
    "Unique Product Names:",
    orders["Product Name"].nunique()
)
print(
    "Product IDs with inconsistent attributes:",
    len(product_id_issues)
)
print(
    "Product Names linked to multiple IDs:",
    (product_ids_per_name > 1).sum()
)


Unique Product IDs: 3788
Unique Product Names: 3788
Product IDs with inconsistent attributes: 0
Product Names linked to multiple IDs: 0


#### Audit Finding: Product Identifier

The dataset contains 3,788 Product IDs and 3,788 Product Names. Every Product ID maps consistently to one Product Name, Category, and Sub-Category, and every Product Name maps to one Product ID.

I will use Product ID as the natural product key when creating DimProduct.

## 11. Text Consistency Audit

I check all text columns for leading or trailing whitespace that could create duplicate categories or failed joins.

In [34]:
datasets_to_check = {
    "Orders": orders,
    "Returns": returns,
    "People": people
}

whitespace_results = []

for dataset_name, dataframe in datasets_to_check.items():
    text_columns = dataframe.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        values = dataframe[column].astype("string")

        whitespace_count = (
            values.notna()
            & (values != values.str.strip())
        ).sum()

        if whitespace_count > 0:
            whitespace_results.append({
                "Dataset": dataset_name,
                "Column": column,
                "Affected Rows": int(whitespace_count)
            })

whitespace_summary = pd.DataFrame(
    whitespace_results
)

display(whitespace_summary)

,Dataset,Column,Affected Rows
0,Orders,Product Name,16


In [35]:
product_name_space_mask = (
    orders["Product Name"]
    != orders["Product Name"].str.strip()
)

whitespace_product_examples = orders.loc[
    product_name_space_mask,
    ["Product ID", "Product Name"]
].drop_duplicates().copy()

whitespace_product_examples["Visible Original"] = (
    whitespace_product_examples["Product Name"].map(repr)
)

whitespace_product_examples["Characters Removed"] = (
    whitespace_product_examples["Product Name"].str.len()
    - whitespace_product_examples["Product Name"]
        .str.strip()
        .str.len()
)

display(
    whitespace_product_examples[
        [
            "Product ID",
            "Visible Original",
            "Characters Removed"
        ]
    ]
)

,Product ID,Visible Original,Characters Removed
1635,TEC-AC-4938,'Kensington SlimBlade Notebook Wireless Mouse ...,1
9147,TEC-AC-3125,'Anker Ultrathin Bluetooth Wireless Keyboard A...,1
10665,OFF-ST-4839,'Iceberg Mobile Mega Data/Printer Cart ',1


#### Audit Finding: Text Whitespace

I found 16 Orders rows across three Product IDs with one trailing space in Product Name. No leading or trailing whitespace was detected in the Returns or People datasets.

During cleaning, I will trim leading and trailing whitespace from text fields while preserving spaces within the text. The original raw files will remain unchanged.

## 12. Categorical Value Audit

I review the distinct values in key categorical fields to identify inconsistent labels before grouping and dashboard development.

In [36]:
categorical_columns = [
    "Ship Mode",
    "Segment",
    "Category",
    "Sub-Category",
    "Order Priority",
    "Market"
]

categorical_results = []

for column in categorical_columns:
    unique_values = sorted(
        orders[column].dropna().unique()
    )

    categorical_results.append({
        "Column": column,
        "Unique Count": len(unique_values),
        "Observed Values": ", ".join(unique_values)
    })

categorical_summary = pd.DataFrame(
    categorical_results
)

display(categorical_summary)

,Column,Unique Count,Observed Values
0,Ship Mode,4,"First Class, Same Day, Second Class, Standard ..."
1,Segment,3,"Consumer, Corporate, Home Office"
2,Category,3,"Furniture, Office Supplies, Technology"
3,Sub-Category,17,"Accessories, Appliances, Art, Binders, Bookcas..."
4,Order Priority,4,"Critical, High, Low, Medium"
5,Market,5,"Africa, Asia Pacific, Europe, LATAM, USCA"


#### Audit Finding: Categorical Values

The core categorical fields contain consistent and interpretable labels. I found no duplicate-looking spellings or capitalization issues requiring category standardization.

## 13. Postal Code Missingness Audit

I determine whether missing Postal Codes represent a data-quality failure or a structural limitation in geographic coverage.

In [37]:
us_rows = orders["Country"] == "United States"

print("United States rows:", us_rows.sum())
print(
    "Missing US Postal Codes:",
    orders.loc[
        us_rows,
        "Postal Code"
    ].isna().sum()
)
print(
    "Non-US rows:",
    (~us_rows).sum()
)
print(
    "Non-US rows with Postal Codes:",
    orders.loc[
        ~us_rows,
        "Postal Code"
    ].notna().sum()
)

valid_us_postal_format = orders.loc[
    us_rows,
    "Postal Code"
].str.fullmatch(
    r"\d{5}",
    na=False
)

print(
    "Invalid US Postal Code formats:",
    (~valid_us_postal_format).sum()
)

United States rows: 9994
Missing US Postal Codes: 0
Non-US rows: 41296
Non-US rows with Postal Codes: 0
Invalid US Postal Code formats: 0


#### Audit Finding: Postal Code Coverage

Postal Code is available for all 9,994 United States sales rows and unavailable for all 41,296 non-US rows. Every available US Postal Code follows a valid five-digit format.

The missing values represent a country-specific coverage limitation rather than random data loss. I will retain all rows, preserve Postal Code as a string field, and leave unavailable non-US values missing.

In [38]:
print(
    "Order Date range:",
    order_date_test.min().date(),
    "to",
    order_date_test.max().date()
)

print(
    "Ship Date range:",
    ship_date_test.min().date(),
    "to",
    ship_date_test.max().date()
)

Order Date range: 2012-01-01 to 2015-12-31
Ship Date range: 2012-01-03 to 2016-01-07


## 14. Dataset Coverage and Analytical Limitations

### Date Coverage

Order Date covers 1 January 2012 through 31 December 2015. Ship Date extends through 7 January 2016 because some year-end orders were shipped after the order year ended.

Revenue and order trends will use Order Date. Shipping analysis will use Ship Date. The future DimDate table must cover both date ranges.

### Field Limitations

- Unit Price is not provided.
- A direct Cost field is not provided.
- Sales and Profit are available at sales-line level.
- Profit Margin can be calculated from Sales and Profit.
- An actual unit selling amount can be calculated as Sales divided by Quantity, but it must not be presented as the original list price.
- Order Status and cancellation information are not available.
- Returns are recorded at order level, without returned product quantities.
- Manager attribution for Canadian sales is unavailable because Orders and People use different regional granularity.

I will limit conclusions to the fields supported by the dataset and will not invent cancellation, product-return quantity, list-price, or cost information.

## 15. Data Quality Audit Summary

The following table consolidates the validated issues, their impact, and the planned cleaning or modeling decisions.

In [39]:
audit_summary = pd.DataFrame([
    {
        "Audit Area": "Missing Values",
        "Finding": (
            f"{orders['Postal Code'].isna().sum():,} missing "
            "Postal Codes; all are non-US records"
        ),
        "Decision": "Retain rows and preserve Postal Code as string"
    },
    {
        "Audit Area": "Exact Duplicates",
        "Finding": (
            f"Orders: {orders.duplicated().sum()}, "
            f"Returns: {returns.duplicated().sum()}, "
            f"People: {people.duplicated().sum()}"
        ),
        "Decision": "No duplicate-row removal required"
    },
    {
        "Audit Area": "Dates and Currency",
        "Finding": (
            f"Invalid dates: "
            f"{order_date_test.isna().sum() + ship_date_test.isna().sum()}; "
            f"invalid Sales/Profit: "
            f"{sales_test.isna().sum() + profit_test.isna().sum()}"
        ),
        "Decision": "Convert dates and currency text to proper types"
    },
    {
        "Audit Area": "Order Identifier",
        "Finding": (
            f"{len(orders_with_multiple_customers)} raw Order ID collisions"
        ),
        "Decision": "Create Order Key from Order ID and Customer ID"
    },
    {
        "Audit Area": "Customer Identifier",
        "Finding": (
            f"{orders['Customer ID'].nunique():,} raw IDs, "
            f"{orders['Customer Name'].nunique():,} names; "
            f"{(customer_ids_per_name > 1).sum():,} fragmented names"
        ),
        "Decision": "Create surrogate Customer Key from Customer Name"
    },
    {
        "Audit Area": "Product Identifier",
        "Finding": (
            f"{len(product_id_issues)} inconsistent Product IDs"
        ),
        "Decision": "Use Product ID as the natural product key"
    },
    {
        "Audit Area": "Text Whitespace",
        "Finding": (
            f"{whitespace_summary['Affected Rows'].sum()} "
            "Product Name rows contain trailing whitespace"
        ),
        "Decision": "Trim leading and trailing text whitespace"
    },
    {
        "Audit Area": "Numeric Validity",
        "Finding": (
            "No invalid Quantity, Sales, Discount, "
            "or Shipping Cost ranges"
        ),
        "Decision": "Retain valid values, including loss transactions"
    },
    {
        "Audit Area": "Statistical Outliers",
        "Finding": "IQR outliers exist across key numeric fields",
        "Decision": "Retain and investigate during business analysis"
    },
    {
        "Audit Area": "Shipping",
        "Finding": (
            f"0 negative durations; "
            f"{len(orders_with_split_shipping)} split-shipping Order Keys"
        ),
        "Decision": "Retain line-level Ship Date and Ship Mode"
    },
    {
        "Audit Area": "Returns Join",
        "Finding": (
            f"{len(returns):,} of {len(returns):,} returns matched "
            "after Canada region standardization"
        ),
        "Decision": "Join using Order ID and standardized Region"
    },
    {
        "Audit Area": "People Mapping",
        "Finding": (
            f"Canada mismatch affects "
            f"{canada_sales_mask.sum():,} sales lines"
        ),
        "Decision": "Do not assign a Canadian manager without a mapping rule"
    }
])

display(audit_summary)

,Audit Area,Finding,Decision
0,Missing Values,"41,296 missing Postal Codes; all are non-US re...",Retain rows and preserve Postal Code as string
1,Exact Duplicates,"Orders: 0, Returns: 0, People: 0",No duplicate-row removal required
2,Dates and Currency,Invalid dates: 0; invalid Sales/Profit: 0,Convert dates and currency text to proper types
3,Order Identifier,11 raw Order ID collisions,Create Order Key from Order ID and Customer ID
4,Customer Identifier,"17,415 raw IDs, 796 names; 795 fragmented names",Create surrogate Customer Key from Customer Name
5,Product Identifier,0 inconsistent Product IDs,Use Product ID as the natural product key
6,Text Whitespace,16 Product Name rows contain trailing whitespace,Trim leading and trailing text whitespace
7,Numeric Validity,"No invalid Quantity, Sales, Discount, or Shipp...","Retain valid values, including loss transactions"
8,Statistical Outliers,IQR outliers exist across key numeric fields,Retain and investigate during business analysis
9,Shipping,0 negative durations; 11 split-shipping Order ...,Retain line-level Ship Date and Ship Mode
